# Sperling (1960) partial report: analysis

Computes the paper's measures from one or more runs' `*_trials.csv`:

- **Experiments 3 and 4** (partial report): per material and cue delay, the percent of the cued row's
  letters reported in their correct positions, and **letters available** = mean correct per report x
  number of rows (Sperling 1960, p. 7). The decay curve over cue delay is Figs. 7 and 8.
- **Experiment 1** (whole report), when a whole-report run is included: letters correctly reported
  per card, the immediate-memory span (the bar at the right of Figs. 7 and 8).
- Checks the paper did not need: the exposure and cue delay measured from the flip stamps, trials
  whose exposure was lengthened by a late flip (excluded), errors by row and by series direction
  (the "observing strategies" of p. 9), and reports that match another row better than the cued one.

Only test trials count: sample trials (`sample == True`) and trials that did not complete are left out,
as are trials whose exposure was not the planned whole frames (`exposure_ok == False`).

Set `ANALYSIS_RUN` to a run folder, or several separated by `:` (e.g. a partial-report run and a whole-report
run of the same subject). By default the newest run under `data/` or `data-rehearsal/` is used.

These numbers describe the run they are computed from. A simulated run's numbers come from the
simulated observer's built-in model, not from anyone's perception.

In [ ]:
import os
from pathlib import Path

import matplotlib

matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

HERE = Path.cwd() if (Path.cwd() / 'run.py').exists() else Path.cwd().parent
RUNS = os.environ.get('ANALYSIS_RUN')
if RUNS:
    runs = [Path(r) for r in RUNS.split(os.pathsep) if r]
else:
    found = sorted(HERE.glob('data*/**/*_trials.csv'), key=lambda p: p.stat().st_mtime)
    runs = [found[-1].parent] if found else []
print('runs:', *runs, sep='\n  ')
trials = pd.concat([pd.read_csv(next(r.glob('*_trials.csv'))).assign(run=r.name) for r in runs],
                   ignore_index=True) if runs else pd.DataFrame()
print(len(trials), 'rows')

In [ ]:
# Which trials count.
if len(trials):
    t = trials.copy()
    t['sample'] = t['sample'].astype(str).str.lower() == 'true'
    done = t['outcome'] == 'REPORTED'
    exposure_ok = t['exposure_ok'].astype(str).str.lower() == 'true'
    summary = {
        'rows': len(t),
        'not completed (re-served)': int((~done).sum()),
        'sample trials': int((done & t['sample']).sum()),
        'exposure lengthened by a late flip (excluded)': int((done & ~t['sample'] & ~exposure_ok).sum()),
    }
    test = t[done & ~t['sample'] & exposure_ok].copy()
    summary['test trials analysed'] = len(test)
    display_table = pd.Series(summary, name='trials')
    print(display_table.to_string())
else:
    test = pd.DataFrame()

In [ ]:
# Timing as shown: exposure (planned whole frames vs flip stamps) and cue delay.
if len(test):
    timing = test.groupby('refresh_hz').agg(
        exposure_frames=('exposure_frames', 'first'),
        exposure_ms_planned=('exposure_ms_planned', 'first'),
        exposure_ms_measured_median=('exposure_ms_measured', 'median'),
        exposure_ms_measured_max=('exposure_ms_measured', 'max'))
    print(timing.round(2).to_string())
    partial = test[test['report'] == 'partial']
    if len(partial):
        cue = partial.groupby('delay_s').agg(
            planned_ms=('cue_delay_ms_planned', 'first'),
            measured_ms_median=('cue_delay_ms_measured', 'median'),
            n=('cue_delay_ms_measured', 'size'))
        print(cue.round(1).to_string())

In [ ]:
# Experiments 3 and 4: percent correct and letters available, by material and cue delay.
partial = test[test['report'] == 'partial'] if len(test) else pd.DataFrame()
if len(partial):
    by_delay = partial.groupby(['material', 'delay_s']).agg(
        n=('n_correct', 'size'),
        mean_correct=('n_correct', 'mean'),
        n_cols=('n_cols', 'first'),
        n_rows=('n_rows', 'first'))
    by_delay['percent_correct'] = 100 * by_delay['mean_correct'] / by_delay['n_cols']
    # Letters available: the partial report as a random sample of what was available (p. 7).
    by_delay['letters_available'] = by_delay['mean_correct'] * by_delay['n_rows']
    by_delay['n_symbols'] = by_delay['n_rows'] * by_delay['n_cols']
    print(by_delay.round(2).to_string())

In [ ]:
# Experiment 1: whole report, the immediate-memory span for the same materials.
whole = test[test['report'] == 'whole'] if len(test) else pd.DataFrame()
span = pd.Series(dtype=float)
if len(whole):
    span = whole.groupby('material')['n_correct'].mean().rename('letters_correct_whole_report')
    print(span.round(2).to_frame().to_string())
else:
    print('no whole-report trials in these runs: run configs/task-whole.yaml for the baseline')

In [ ]:
# The decay curve (Figs. 7 and 8): letters available against the delay of the cue,
# percent correct on the right axis, the whole-report span as a bar at the right.
if len(partial):
    materials = list(by_delay.index.get_level_values('material').unique())
    fig, axes = plt.subplots(1, len(materials), figsize=(4.2 * len(materials), 3.6), squeeze=False)
    for ax, m in zip(axes[0], materials, strict=True):
        d = by_delay.loc[m]
        n_sym = int(d['n_symbols'].iloc[0])
        ax.plot(d.index, d['letters_available'], 'o-', color='black')
        ax.axvline(0, ls='--', color='grey', lw=0.8)
        ax.axvspan(-0.05, 0.0, color='0.85', lw=0)  # the 50 ms exposure, ending at 0
        if m in span.index:
            ax.bar([d.index.max() + 0.15], [span[m]], width=0.05, color='black')
        ax.set_ylim(0, n_sym)
        ax.set_xlabel('delay of the cue from the array offset (s)')
        ax.set_ylabel('letters available')
        ax.set_title(f'{m} ({n_sym} symbols)')
        right = ax.twinx()
        right.set_ylim(0, 100)
        right.set_ylabel('percent correct')
    fig.tight_layout()
    plt.show()

In [ ]:
# Errors by cued row and by series direction (the strategies of p. 9: equal attention vs
# anticipating one row shows as a row preference, and as a difference between series).
if len(partial):
    by_row = partial.groupby(['material', 'cued_row'])['prop_correct'].agg(['mean', 'size'])
    print(by_row.round(3).to_string())
    by_series = partial.groupby(['material', 'series_direction', 'delay_s'])['prop_correct'].mean()
    print(by_series.unstack('delay_s').round(3).to_string())

In [ ]:
# Accuracy by position in the row, and reports that match another row better than the cued
# one (the paper asked an observer who wrote the wrong row to write the right one, p. 7;
# here they can only be counted).
if len(partial):
    masks = partial['correct_mask'].astype(str).str.zfill(int(partial['n_cols'].max()))
    for m, g in partial.assign(mask=masks).groupby('material'):
        n = int(g['n_cols'].iloc[0])
        pos = [g['mask'].str[-n:].str[i].astype(int).mean() for i in range(n)]
        print(m, 'by position:', np.round(pos, 3))
    wrong_row = partial['other_row_max_correct'] > partial['n_correct']
    print('reports closer to another row than to the cued one:', int(wrong_row.sum()), 'of', len(partial))